# 🎬 Dự án: Xây dựng Data Lakehouse & Churn Prediction cho nền tảng Streaming

## 📌 Giới thiệu
Trong bối cảnh các nền tảng streaming cạnh tranh ngày càng khốc liệt, việc **giữ chân người dùng (customer retention)** trở thành yếu tố sống còn.

Dự án này tập trung vào bài toán **dự đoán churn (người dùng rời bỏ hệ thống)** dựa trên hành vi và mức độ tương tác.

---

## 🎯 Mục tiêu dự án
- Xây dựng kiến trúc **Data Lakehouse 3 tầng**:
  - 🥉 Bronze: Dữ liệu thô
  - 🥈 Silver: Dữ liệu đã làm sạch
  - 🥇 Gold: Dữ liệu phục vụ Machine Learning

- Xây dựng mô hình dự đoán churn:
  - Target: `is_active`
  - Bài toán: Classification

---

## 📊 Dataset sử dụng
Bao gồm các bảng:
- users
- movies
- watch_history
- recommendation_logs
- search_logs
- reviews

---

## 🚀 Quy trình thực hiện
1. Load dữ liệu vào Bronze Layer  
2. Làm sạch dữ liệu (Silver Layer)  
3. Feature Engineering (Gold Layer)  
4. Train & Evaluate model  

# 🥉 Bronze Layer: Data Ingestion

Trong bước này, chúng ta load dữ liệu thô từ Workspace vào hệ thống và lưu dưới dạng bảng Delta.

Dữ liệu vẫn giữ nguyên trạng thái ban đầu (chưa xử lý missing values, duplicates, outliers).

In [0]:
# Base path
base_path = "/Volumes/workspace/netflix/netflix/"

# Load tables

# Load users
users_raw = spark.read.csv(
    base_path + "users.csv",
    header=True,
    inferSchema=True
)

# Load movies
movies_raw = spark.read.csv(
    base_path + "movies.csv",
    header=True,
    inferSchema=True
)

# Load watch history
watch_history_raw = spark.read.csv(
    base_path + "watch_history.csv",
    header=True,
    inferSchema=True
)

# Load recommendation logs
recommendation_logs_raw = spark.read.csv(
    base_path + "recommendation_logs.csv",
    header=True,
    inferSchema=True
)

# Load search logs
search_logs_raw = spark.read.csv(
    base_path + "search_logs.csv",
    header=True,
    inferSchema=True
)

# Load reviews
reviews_raw = spark.read.csv(
    base_path + "reviews.csv",
    header=True,
    inferSchema=True
)

# Set catalog & schema
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA netflix")

# 💾 SAVE Bronze tables (THIS WAS MISSING)
users_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_users")
movies_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_movies")
watch_history_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_watch_history")
recommendation_logs_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_recommendation_logs")
search_logs_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_search_logs")
reviews_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_reviews")

### 👀 Kiểm tra dữ liệu ban đầu

Sau khi load dữ liệu, chúng ta cần kiểm tra nhanh:
- Schema (kiểu dữ liệu)
- Một vài dòng dữ liệu mẫu
- Phát hiện sơ bộ vấn đề (missing values, format,...)

In [0]:
# Xem dữ liệu users
display(users_raw)

# Xem dữ liệu watch history
display(watch_history_raw)

### 🔍 Kiểm tra schema
users_raw.printSchema()
watch_history_raw.printSchema()
movies_raw.printSchema()

### ⚠️ Kiểm tra missing values

Dữ liệu thực tế thường không hoàn hảo:
- Thiếu dữ liệu (missing)
- Không đồng nhất

👉 Ở Bronze Layer: chỉ QUAN SÁT, chưa xử lý

In [0]:
from pyspark.sql.functions import col, sum

def check_missing(df):
    return df.select([
        sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
    ])

display(check_missing(users_raw))
print("Users count:", users_raw.count())
print("Users after dedup:", users_raw.dropDuplicates().count())

display(check_missing(watch_history_raw))
print("Watch history count:", watch_history_raw.count())
print("Watch history after dedup:", watch_history_raw.dropDuplicates().count())

### 📊 Nhận xét dữ liệu Bronze

- Bảng users có nhiều missing values ở các cột quan trọng (age, monthly_spend,...)
- Tồn tại khoảng 300 bản ghi trùng lặp (~3%)

Đối với các bảng còn lại, chúng ta thực hiện kiểm tra nhanh:
- Số lượng bản ghi
- Số lượng bản ghi trùng lặp

In [0]:
def quick_summary(df, name):
    total = df.count()
    dedup = df.dropDuplicates().count()
    
    print(f"Table: {name}")
    print(f"Total rows: {total}")
    print(f"After dedup: {dedup}")
    print(f"Duplicates: {total - dedup}")
    print("-" * 40)

quick_summary(movies_raw, "movies")
quick_summary(recommendation_logs_raw, "recommendation_logs")
quick_summary(search_logs_raw, "search_logs")
quick_summary(reviews_raw, "reviews")

### 📌 Nhận xét tổng quan

- Các bảng đều có tồn tại duplicate records
- Một số bảng có missing values (đã kiểm tra chi tiết ở users và watch_history)
- Dữ liệu phản ánh đúng đặc điểm dữ liệu thực tế: không hoàn hảo

👉 Các vấn đề này sẽ được xử lý ở Silver Layer

# 🥈 Silver Layer: Data Cleaning & Preprocessing

Ở bước này, chúng ta sẽ:
- Loại bỏ duplicate
- Xử lý missing values
- Chuẩn hóa dữ liệu

👉 Áp dụng cho TẤT CẢ các bảng để đảm bảo dữ liệu sạch và nhất quán

### 📥 Load dữ liệu từ Bronze Layer

Chúng ta không làm việc trực tiếp với CSV nữa, mà sử dụng các bảng Bronze đã lưu.

👉 Điều này thể hiện đúng kiến trúc Lakehouse (data reuse, pipeline rõ ràng)

In [0]:
users_bronze = spark.table("bronze_users")
watch_bronze = spark.table("bronze_watch_history")
movies_bronze = spark.table("bronze_movies")
recommendation_logs_bronze = spark.table("bronze_recommendation_logs")
search_logs_bronze = spark.table("bronze_search_logs")
reviews_bronze = spark.table("bronze_reviews")

## 👤 Cleaning bảng USERS

Đây là bảng quan trọng nhất cho bài toán churn prediction

In [0]:
from pyspark.sql.functions import col, when
# Remove duplicates
users_silver = users_bronze.dropDuplicates(["email"])

# Median age
median_age = users_silver.approxQuantile("age", [0.5], 0.01)[0]

users_silver = users_silver.withColumn(
    "age",
    when(col("age").isNull(), median_age).otherwise(col("age"))
)

# Gender
users_silver = users_silver.withColumn(
    "gender",
    when(col("gender").isNull(), "Unknown").otherwise(col("gender"))
)

# Monthly spend
median_spend = users_silver.approxQuantile("monthly_spend", [0.5], 0.01)[0]

users_silver = users_silver.withColumn(
    "monthly_spend",
    when(col("monthly_spend").isNull(), median_spend).otherwise(col("monthly_spend"))
)

# Household size
median_household = users_silver.approxQuantile("household_size", [0.5], 0.01)[0]

users_silver = users_silver.withColumn(
    "household_size",
    when(col("household_size").isNull(), median_household).otherwise(col("household_size"))
)

# Remove basic outliers
users_silver = users_silver.filter(
    (col("age") >= 10) & (col("age") <= 100)
)

# Chuẩn hóa dữ liệu dạng categorical
users_silver = users_silver.withColumn(
    "gender",
    when(col("gender").isNull(), "Unknown")
    .otherwise(col("gender"))
)

# Lowercase for consistency
from pyspark.sql.functions import lower

users_silver = users_silver.withColumn("gender", lower(col("gender")))
users_silver = users_silver.withColumn("subscription_plan", lower(col("subscription_plan")))

## 📺 Cleaning bảng WATCH HISTORY

Bảng này rất quan trọng vì chứa hành vi xem phim của user

In [0]:
# Remove duplicates
watch_silver = watch_bronze.dropDuplicates()

# Fill important numeric columns ONLY
median_duration = watch_silver.approxQuantile("watch_duration_minutes", [0.5], 0.01)[0]
median_progress = watch_silver.approxQuantile("progress_percentage", [0.5], 0.01)[0]

watch_silver = watch_silver.withColumn(
    "watch_duration_minutes",
    when(col("watch_duration_minutes").isNull(), median_duration)
    .otherwise(col("watch_duration_minutes"))
)

watch_silver = watch_silver.withColumn(
    "progress_percentage",
    when(col("progress_percentage").isNull(), median_progress)
    .otherwise(col("progress_percentage"))
)

# Remove invalid values
watch_silver = watch_silver.filter(
    (col("progress_percentage") >= 0) & (col("progress_percentage") <= 100)
)

# IMPORTANT: Convert rating into behavior instead of filling
watch_silver = watch_silver.withColumn(
    "has_rating",
    when(col("user_rating").isNull(), 0).otherwise(1)
)

# Drop noisy column
watch_silver = watch_silver.drop("user_rating")

## 🎥 Cleaning bảng MOVIES

In [0]:
# Remove duplicates
movies_silver = movies_bronze.dropDuplicates()

# Fill only meaningful fields
median_rating = movies_silver.approxQuantile("imdb_rating", [0.5], 0.01)[0]

movies_silver = movies_silver.withColumn(
    "imdb_rating",
    when(col("imdb_rating").isNull(), median_rating)
    .otherwise(col("imdb_rating"))
)

# Add structural feature
movies_silver = movies_silver.withColumn(
    "is_series",
    when(col("content_type") == "TV Series", 1).otherwise(0)
)

# Fill safe categorical only
movies_silver = movies_silver.fillna({
    "genre_secondary": "unknown"
})

# ⚠ DO NOT fill budget / revenue / episodes → keep NULL

%md
## 🤖 Cleaning sơ bộ các bảng phụ

In [0]:
# Remove duplicates

# Cleaning bảng RECOMMENDATION_LOGS
# Remove duplicates
recommendation_logs_silver = recommendation_logs_bronze.dropDuplicates()

# Light handling only
recommendation_logs_silver = recommendation_logs_silver.withColumn(
    "recommendation_score",
    when(col("recommendation_score").isNull(), 0.5)
    .otherwise(col("recommendation_score"))
)

recommendation_logs_silver = recommendation_logs_silver.fillna({
    "algorithm_version": "unknown"
})

# Cleaning bảng SEARCH_LOGS
# Remove duplicates
search_logs_silver = search_logs_bronze.dropDuplicates()

# Convert missing click → behavior
search_logs_silver = search_logs_silver.withColumn(
    "clicked",
    when(col("clicked_result_position").isNull(), 0).otherwise(1)
)

# Drop original column (too sparse)
search_logs_silver = search_logs_silver.drop("clicked_result_position")

# Cleaning bảng REVIEWS
# Remove duplicates
reviews_silver = reviews_bronze.dropDuplicates()

# Fill only numeric safely
reviews_silver = reviews_silver.fillna({
    "helpful_votes": 0,
    "total_votes": 0,
    "sentiment_score": 0.5
})

# Drop heavy text (not needed for churn)
reviews_silver = reviews_silver.drop("review_text")


### 💾 Lưu dữ liệu Silver Layer

Sau bước này:
- Dữ liệu đã sạch
- Có thể reuse cho nhiều mục đích

In [0]:
users_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_users")

watch_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_watch_history")

movies_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_movies")

recommendation_logs_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_recommendation_logs")

search_logs_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_search_logs")

reviews_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_reviews")

### 🧪 Data Quality Check (Silver Layer)

Mục tiêu:
- Đánh giá chất lượng dữ liệu sau khi làm sạch
- Áp dụng thống nhất cho tất cả các bảng

Kiểm tra:
- Missing values
- Duplicate records
- Thống kê cơ bản

In [0]:
from pyspark.sql import functions as F

def data_quality_report(df, name, key_cols=None):
    print(f"\n📊 TABLE: {name}")
    print("=" * 60)

    # Row count
    total = df.count()
    print(f"Total rows: {total}")

    # Duplicate rows
    total_distinct = df.dropDuplicates().count()
    print(f"Exact duplicate rows: {total - total_distinct}")

    # Key-based duplicates
    if key_cols:
        key_dups = df.groupBy(key_cols).count().filter(F.col("count") > 1).count()
        print(f"Duplicate groups by {key_cols}: {key_dups}")

    # Missing values (null + NaN for numeric columns)
    print("\nMissing values:")
    missing_exprs = []
    for c, t in df.dtypes:
        if t in ("double", "float", "int", "bigint", "decimal"):
            missing_exprs.append(
                F.sum(
                    F.when(F.col(c).isNull() | F.isnan(F.col(c)), 1).otherwise(0)
                ).alias(c)
            )
        else:
            missing_exprs.append(
                F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
            )

    df.select(missing_exprs).show(truncate=False)

    # Basic stats for numeric columns only
    numeric_cols = [c for c, t in df.dtypes if t in ("double", "float", "int", "bigint", "decimal")]
    if numeric_cols:
        print("\nBasic statistics:")
        df.select(numeric_cols).describe().show()

    print("\nSample rows:")
    df.show(5, truncate=False)

    print("=" * 60)

data_quality_report(users_silver, "users_silver", key_cols=["user_id"])
data_quality_report(watch_silver, "watch_silver", key_cols=["session_id"])
data_quality_report(movies_silver, "movies_silver", key_cols=["movie_id"])
data_quality_report(recommendation_logs_silver, "recommendation_logs_silver", key_cols=["recommendation_id"])
data_quality_report(search_logs_silver, "search_logs_silver", key_cols=["search_id"])
data_quality_report(reviews_silver, "reviews_silver", key_cols=["review_id"])

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.netflix.export;

In [0]:
base_export_path = "/Volumes/workspace/netflix/export"

# USERS
spark.table("silver_users").write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_export_path}/silver_users_csv")

# WATCH HISTORY
spark.table("silver_watch_history").write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_export_path}/silver_watch_history_csv")

# MOVIES
spark.table("silver_movies").write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_export_path}/silver_movies_csv")

# RECOMMENDATION LOGS
spark.table("silver_recommendation_logs").write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_export_path}/silver_recommendation_logs_csv")

# SEARCH LOGS
spark.table("silver_search_logs").write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_export_path}/silver_search_logs_csv")

# REVIEWS
spark.table("silver_reviews").write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(f"{base_export_path}/silver_reviews_csv")